In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

In [2]:
model_name = 'bert'
device = 'cuda'

In [3]:

# load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.to(device)  # move model to specified device

d:\ProgramData\anaconda3\envs\t\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [4]:
comments = pd.read_csv('./imdb_sup.csv')
def rate2label(rate):
    """
    1-10,Three classes
    :param rate:
    :return:
    """
    if rate <= 3:
        return 0
    elif rate <= 6:
        return 1
    else:
        return 2
comments['Rating'] = comments['Rating'].apply(lambda x: rate2label(x))

In [5]:
all_comments = comments['Review'].values.tolist()
all_labels = comments['Rating'].values.tolist()
str_comments = [str(c) for c in all_comments]
select_idx = np.random.choice(len(str_comments), 10000, replace=False)
str_comments = [str_comments[i] for i in select_idx]
all_labels = [all_labels[i] for i in select_idx]

In [6]:
# create linear layer, to reduce dimension
import torch.nn as nn
linear_layer = nn.Linear(768, 128).to(device)
def get_bert_embeddings(text):
    # ncode the input text
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True,max_length=512)
    #  move encoded input to specified device
    encoded_input = {key: val.to(device) for key, val in encoded_input.items()}
    # obtain  BERT model output
    with torch.no_grad():  
        output = model(**encoded_input)
        #  extract the hidden state of the last layer
        last_hidden_states = linear_layer(output.last_hidden_state)
    return last_hidden_states.squeeze(0)

In [7]:
def pad_and_truncate(rep:torch.Tensor, max_len=128):
    length,dim = rep.shape
    if length > max_len:
        return rep[:max_len,:]
    else:
        return torch.cat((rep,torch.zeros(max_len-length,dim)),dim=0)

In [8]:
bert_rep = []
for s in tqdm(str_comments):
    rep =get_bert_embeddings(s).cpu()
    rep = pad_and_truncate(rep)
    bert_rep.append(rep)


100%|██████████| 10000/10000 [04:01<00:00, 41.47it/s]


In [9]:
bert_rep = np.array([i.numpy() for i in tqdm(bert_rep)])
np.savez('./bert_rep.npz',bert_rep=bert_rep,all_labels=all_labels)

100%|██████████| 10000/10000 [00:00<00:00, 285655.21it/s]
